# 11 - LlamaIndex Governed Retrieval Pattern

This notebook demonstrates a LlamaIndex-style retrieval tool. The retrieval function searches policy context, then checks Metatate before returning data-bearing context to the agent.

If LlamaIndex is installed, the same function can be wrapped as a `FunctionTool`. Without LlamaIndex, the notebook runs the function directly.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## A small policy-context retriever


In [ ]:
policy_dir = repo_root / "sample-data" / "acmecloud" / "policies"
policy_docs = [
    {"name": path.name, "text": path.read_text(encoding="utf-8")}
    for path in sorted(policy_dir.glob("*.yaml"))
]

def keyword_score(text, query):
    terms = [term.lower() for term in query.split() if len(term) > 3]
    lowered = text.lower()
    return sum(lowered.count(term) for term in terms)

def retrieve_policy_context(query, limit=2):
    scored = sorted(
        (
            {**doc, "score": keyword_score(doc["text"], query)}
            for doc in policy_docs
        ),
        key=lambda item: item["score"],
        reverse=True,
    )
    return scored[:limit]


## Govern the retrieval answer before returning it


In [ ]:
def governed_retrieval_answer(query):
    retrieved = retrieve_policy_context(query)
    sql = "SELECT region, account_status, SUM(arr) AS arr FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE account_status = 'active' GROUP BY region, account_status"
    validation = client.validate_query_context(
        sql,
        operation="read",
        intended_use="analytics",
        actor_role="DATA_ANALYST",
    )
    decision = decision_label(validation)
    if decision == "DENY":
        return {
            "decision": decision,
            "answer": "Metatate blocked data-bearing context for this request.",
            "sources": [],
            "metatate_action": agent_action_text(validation),
        }
    return {
        "decision": decision,
        "answer": "Use aggregate ARR by region and account status. Do not include customer identifiers.",
        "sources": [item["name"] for item in retrieved],
        "metatate_action": agent_action_text(validation),
    }

query = "What governed context can an analyst use for active customer ARR by region?"
answer = governed_retrieval_answer(query)
print(json.dumps(answer, indent=2))


## Optional LlamaIndex wrapper


In [ ]:
try:
    from llama_index.core.tools import FunctionTool

    tool = FunctionTool.from_defaults(fn=governed_retrieval_answer)
    print("Wrapped governed_retrieval_answer as a LlamaIndex FunctionTool.")
    print(tool.metadata.name)
except ImportError:
    print("LlamaIndex is not installed. The governed retrieval function above is the same callable you would wrap as a FunctionTool.")


The key point is placement: retrieval is not allowed to bypass governance. The retriever returns only context that survives a Metatate decision check.
